# Métriques d'évaluation du système de Fact-Checking

L'évaluation de notre système ne doit pas se limiter à mesurer si le modèle prédit correctement le label d'une affirmation. Un système de fact-checking fondé sur les preuves doit également être capable de retrouver les bonnes preuves et d'associer à ses prédictions un niveau de confiance fiable.

Pour cette raison, nous retenons quatre métriques principales :

1. **FEVER Score** : évalue simultanément la classification et la justification par les preuves ;
2. **Macro-F1** : évalue la qualité de la classification des trois labels ;
3. **Evidence Precision et Evidence Recall** : évaluent la qualité du système de récupération des preuves ;
4. **Expected Calibration Error (ECE)** : évalue la fiabilité des scores de confiance du modèle.

---

# 1. FEVER Score

## Objectif

Le **FEVER Score** est la métrique la plus importante pour notre système, car elle évalue la réussite complète d'une prédiction.

Pour qu'une prédiction soit considérée comme correcte, deux conditions doivent être satisfaites simultanément :

* le système doit prédire le bon label ;
* le système doit fournir au moins un ensemble de preuves correct permettant de justifier cette prédiction.

Ainsi, une classification correcte mais accompagnée de mauvaises preuves n'est pas considérée comme une réussite complète.

## Exemple

Considérons l'affirmation suivante :

> **Claim :** "The Earth is flat."

La vérité attendue est :

```text
Label : REFUTES
Evidence : [article, sentence]
```

Notre système produit :

```text
Label : REFUTES
Evidence : mauvaise phrase
```

L'évaluation donne alors :

```text
Classification correcte     ✓
Preuve correcte             ✗
FEVER Score                 ✗
```

Le système a correctement identifié que l'affirmation devait être réfutée, mais il n'a pas fourni la preuve permettant de justifier cette décision.

Le FEVER Score est donc particulièrement adapté à notre projet, car l'objectif du système n'est pas simplement :

> « Prédire si une affirmation est vraie ou fausse. »

L'objectif est plutôt :

> **« Déterminer la véracité d'une affirmation et justifier cette décision à l'aide de preuves vérifiables. »**

Le FEVER Score évalue donc directement la réussite de cette tâche complète.

On peut le représenter conceptuellement comme suit :

```text
Prédiction correcte
        +
Preuve correcte
        ↓
   FEVER Score = 1
```

Si l'une de ces deux conditions n'est pas satisfaite :

```text
Label correct + preuve incorrecte
        ↓
   FEVER Score = 0
```

Le FEVER Score permet ainsi d'évaluer le système de manière plus réaliste qu'une simple métrique de classification.

---

# 2. Macro-F1

## Objectif

Le module de vérification doit classer chaque affirmation dans l'une des trois catégories du dataset FEVER :

```text
SUPPORTS
REFUTES
NOT ENOUGH INFO
```

Pour chacune de ces classes, nous calculons un score F1 individuel :

```text
F1_SUPPORTS
F1_REFUTES
F1_NEI
```

Le **Macro-F1** correspond ensuite à la moyenne arithmétique de ces trois scores :

```text
Macro-F1 =
(F1_SUPPORTS + F1_REFUTES + F1_NEI) / 3
```

Le score F1 combine deux notions importantes :

* la **Precision** : parmi les prédictions d'une classe, combien sont correctes ?
* le **Recall** : parmi les exemples réellement appartenant à cette classe, combien ont été correctement identifiés ?

Le F1-score combine ces deux mesures à l'aide de leur moyenne harmonique :

```text
F1 = 2 × (Precision × Recall)
     -------------------------
       Precision + Recall
```

## Pourquoi utiliser le Macro-F1 ?

Le dataset FEVER présente une distribution déséquilibrée entre les classes :

```text
SUPPORTS          ≈ 80 035 exemples
NOT ENOUGH INFO   ≈ 35 639 exemples
REFUTES           ≈ 29 775 exemples
```

La classe **SUPPORTS** est donc beaucoup plus représentée que les autres.

Dans ce contexte, l'accuracy peut donner une impression exagérément positive des performances du modèle.

Par exemple :

```text
Accuracy = 80 %
```

Ce résultat ne signifie pas nécessairement que le modèle reconnaît correctement les trois catégories. Il pourrait obtenir une bonne accuracy en étant particulièrement performant sur la classe majoritaire, tout en ayant de mauvaises performances sur **REFUTES** ou **NOT ENOUGH INFO**.

Le Macro-F1 permet de limiter ce problème, car chaque classe possède exactement le même poids dans le calcul final :

```text
SUPPORTS          → 1/3 du score
REFUTES           → 1/3 du score
NOT ENOUGH INFO   → 1/3 du score
```

Cette métrique nous permet donc de vérifier que le modèle est équilibré dans sa capacité à distinguer les trois types de verdicts.

---

# 3. Evidence Precision et Evidence Recall

Ces métriques évaluent spécifiquement la capacité du système à retrouver les preuves pertinentes.

Dans notre architecture, le processus peut être représenté ainsi :

```text
Claim
  ↓
Retrieval
  ↓
FAISS
  ↓
Reranker
  ↓
Evidence sélectionnée
```

L'objectif est de comparer les preuves produites par notre système avec les preuves annotées comme correctes dans le dataset FEVER.

---

## 3.1 Evidence Recall

L'Evidence Recall répond à la question suivante :

> **Parmi les preuves correctes disponibles dans FEVER, combien notre système a-t-il réussi à retrouver ?**

Supposons que les preuves correctes soient :

```text
{E1, E2}
```

Et que notre système récupère :

```text
{E1, E3, E4}
```

Le système a retrouvé E1, mais pas E2.

Nous avons donc :

```text
Nombre de preuves correctes retrouvées = 1
Nombre total de preuves correctes       = 2
```

Ainsi :

```text
Evidence Recall = 1 / 2 = 50 %
```

Le Recall mesure donc la capacité du système à **ne pas manquer les preuves importantes**.

Un Recall élevé signifie que le système retrouve une grande partie des preuves nécessaires à la vérification.

---

## 3.2 Evidence Precision

L'Evidence Precision répond à une question différente :

> **Parmi toutes les preuves récupérées par le système, combien sont réellement correctes ?**

Dans l'exemple précédent :

```text
Preuves récupérées :
{E1, E3, E4}

Preuves réellement correctes :
{E1, E2}
```

Une seule preuve récupérée est correcte : E1.

Nous avons donc :

```text
Nombre de preuves correctes récupérées = 1
Nombre total de preuves récupérées     = 3
```

Ainsi :

```text
Evidence Precision = 1 / 3 ≈ 33 %
```

La Precision mesure donc la capacité du système à **éviter de récupérer des preuves incorrectes ou non pertinentes**.

---

## La différence fondamentale

La distinction entre les deux métriques peut être résumée ainsi :

```text
RECALL
   ↓
Parmi toutes les bonnes preuves,
combien ai-je réussi à retrouver ?
```

```text
PRECISION
   ↓
Parmi les preuves que j'ai retrouvées,
combien sont réellement correctes ?
```

Un système peut donc avoir :

```text
Recall élevé
+
Precision faible
```

Cela signifie qu'il retrouve la plupart des bonnes preuves, mais qu'il récupère également beaucoup de preuves incorrectes.

À l'inverse :

```text
Precision élevée
+
Recall faible
```

signifie que les preuves récupérées sont généralement correctes, mais que le système manque une grande partie des preuves disponibles.

---

## Évaluation du Retriever et du Reranker

Ces métriques sont particulièrement importantes pour notre architecture, car elles nous permettent d'évaluer séparément les différentes étapes du système.

Nous pouvons comparer :

```text
Claim
  ↓
FAISS Retriever
  ↓
Evidence
  ↓
Precision / Recall
```

Puis :

```text
Claim
  ↓
FAISS Retriever
  ↓
Reranker
  ↓
Evidence sélectionnée
  ↓
Precision / Recall
```

La comparaison des résultats nous permettra de déterminer objectivement si le reranker améliore réellement la qualité des preuves.

Par exemple :

| Système          | Evidence Precision | Evidence Recall |
| ---------------- | -----------------: | --------------: |
| FAISS seul       |               35 % |            78 % |
| FAISS + Reranker |               62 % |            75 % |

Dans cet exemple, le reranker améliore fortement la précision des preuves, mais entraîne une légère diminution du Recall.

Cette analyse est essentielle pour comprendre le comportement réel de notre système et mesurer l'apport du reranking au-delà de la simple classification finale.

---

# 4. ECE — Expected Calibration Error

## Objectif

Le **Expected Calibration Error (ECE)** mesure la fiabilité des probabilités produites par le modèle.

Supposons que notre système produise la prédiction suivante :

```text
Label      : SUPPORTS
Confidence : 0.95
```

Une confiance de 95 % devrait signifier que, sur un grand nombre de prédictions auxquelles le modèle attribue une confiance proche de 95 %, environ 95 % sont effectivement correctes.

Autrement dit :

```text
Confiance annoncée ≈ Probabilité réelle de réussite
```

## Exemple d'un modèle mal calibré

Supposons que le système réalise :

```text
Nombre de prédictions      : 100
Confiance moyenne          : 90 %
Accuracy réelle            : 70 %
```

Le modèle annonce donc une confiance de 90 %, alors qu'il n'est correct que dans 70 % des cas.

Il est alors **trop confiant**.

```text
Confiance : 90 %
Accuracy  : 70 %
      ↓
Mauvaise calibration
```

L'ECE mesure précisément l'écart entre :

* la confiance moyenne attribuée par le modèle ;
* la proportion réelle de prédictions correctes.

De manière simplifiée :

```text
ECE ≈ Écart moyen entre
      confiance prédite
      et exactitude observée
```

Un modèle parfaitement calibré aurait idéalement :

```text
Confidence = 0.90
        ↓
Accuracy réelle ≈ 90 %
```

---

## Interprétation

```text
ECE faible
    ↓
Les probabilités sont fiables
    ↓
Une confiance de 90 % correspond approximativement
à une probabilité réelle de réussite de 90 %
```

À l'inverse :

```text
ECE élevé
    ↓
Le modèle est mal calibré
    ↓
Les scores de confiance ne reflètent pas
correctement la fiabilité réelle du modèle
```

---

## Pourquoi l'ECE est important pour un fact-checker ?

Dans un système de fact-checking, le score de confiance peut être utilisé pour :

* signaler automatiquement les affirmations très probablement correctes ou fausses ;
* transmettre les cas incertains à un humain ;
* prioriser les vérifications ;
* afficher un niveau de confiance à l'utilisateur ;
* prendre des décisions automatisées.

Il est donc important qu'une valeur telle que :

```text
Confidence = 0.98
```

ne signifie pas simplement :

> « Le modèle semble presque sûr de lui. »

Elle doit plutôt correspondre à une véritable fiabilité statistique :

> **« Parmi les prédictions auxquelles le modèle attribue environ 98 % de confiance, la grande majorité doit effectivement être correcte. »**

Un modèle peut donc avoir une bonne accuracy tout en étant mal calibré.

Par exemple :

```text
Accuracy élevée
+
ECE élevé
```

signifie que le modèle prédit souvent correctement, mais que ses probabilités de confiance sont peu fiables.

À l'inverse :

```text
Accuracy élevée
+
ECE faible
```

indique un système à la fois performant et fiable dans l'interprétation de ses propres prédictions.

---

# Synthèse des métriques

Chaque métrique évalue une dimension différente de notre système :

| Métrique               | Élément évalué         | Question principale                                        |
| ---------------------- | ---------------------- | ---------------------------------------------------------- |
| **FEVER Score**        | Système complet        | Le label et les preuves sont-ils tous les deux corrects ?  |
| **Macro-F1**           | Module de vérification | Le modèle distingue-t-il correctement les trois classes ?  |
| **Evidence Precision** | Retrieval / Reranking  | Les preuves récupérées sont-elles réellement pertinentes ? |
| **Evidence Recall**    | Retrieval / Reranking  | Le système retrouve-t-il les preuves nécessaires ?         |
| **ECE**                | Calibration            | Les scores de confiance sont-ils fiables ?                 |

L'utilisation conjointe de ces métriques permet donc d'obtenir une évaluation complète :

```text
                    ┌────────────────────┐
                    │      CLAIM         │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │     RETRIEVAL      │
                    │       FAISS        │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │      RERANKER      │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │     EVIDENCE       │
                    └─────────┬──────────┘
                              ↓
                    ┌────────────────────┐
                    │    VERIFICATION    │
                    │ SUPPORTS / REFUTES│
                    │     / NEI          │
                    └────────────────────┘
```

L'évaluation peut alors être organisée de la manière suivante :

```text
Retriever + Reranker
        ↓
Evidence Precision
Evidence Recall
        ↓
        ↓
   Evidence correcte
        +
Classification correcte
        ↓
    FEVER Score

Classification seule
        ↓
     Macro-F1

Prédiction + confiance
        ↓
        ECE
```

Cette combinaison est particulièrement pertinente pour notre projet, car elle permet d'évaluer non seulement si le système donne la bonne réponse, mais également :

* s'il retrouve les bonnes preuves ;
* s'il évite les preuves incorrectes ;
* s'il est équilibré entre les différentes classes ;
* et si son niveau de confiance reflète réellement sa fiabilité.

Ainsi, l'évaluation ne se limite pas à mesurer la performance d'un classifieur. Elle permet d'évaluer l'ensemble de la chaîne de fact-checking :

> **Recherche → Reranking → Sélection des preuves → Vérification → Confiance de la prédiction.**
